# Episode 10 — Reranking: The Single Biggest Quality Upgrade

> *'Retrieval finds candidates. Reranking picks the winner.'*

In [ ]:
import sys,json
from pathlib import Path
cwd=Path().resolve(); repo_root=cwd.parent if cwd.name=='notebooks' else cwd
sys.path.insert(0,str(repo_root/'src'))
from dotenv import load_dotenv; load_dotenv(repo_root/'.env',override=True)
print('ready')

## 1. Why reranking matters

Vector search ranks by cosine similarity — a weak proxy for relevance.
A cross-encoder reads the full (query, document) pair and scores true relevance.

```
Vector:  query_embedding · doc_embedding  (128ms for 10k docs)
Rerank:  cross_encoder(query, doc)        (runs on top-20 only)
```

Running cross-encoder on all chunks is too slow.
Running it on top-20 from vector search is fast AND accurate.

In [ ]:
import sys,json
from pathlib import Path
cwd=Path().resolve(); repo_root=cwd.parent if cwd.name=='notebooks' else cwd
sys.path.insert(0,str(repo_root/'src'))
from dotenv import load_dotenv; load_dotenv(repo_root/'.env',override=True)
print('ready')

In [ ]:
# Two-stage pipeline: vector top-20 → rerank to top-5
from rag.retrieval.vector_retriever import VectorRetriever
from rag.retrieval.reranker import Reranker, RerankerBackend
retriever = VectorRetriever()
reranker  = Reranker(backend=RerankerBackend.COHERE)
query = 'What is the contraceptive prevalence in Nigeria?'
docs_vector  = retriever.retrieve(query, top_k=20)
docs_reranked = reranker.rerank(query, docs_vector, top_n=5)
print(f'Vector top-20 → Reranked top-5')
print()
print('BEFORE reranking (top-5 by cosine):')
for d in docs_vector[:5]:
    print(f'  [{d.metadata["similarity_score"]:.3f}] {d.metadata.get("country","")} | {d.page_content[:100]}...')
print()
print('AFTER reranking (top-5 by Cohere):')
for d in docs_reranked:
    print(f'  [{d.metadata["rerank_score"]:.3f}] {d.metadata.get("country","")} | {d.page_content[:100]}...')

In [ ]:
# Local cross-encoder alternative (no API cost)
from rag.retrieval.reranker import Reranker, RerankerBackend
local_reranker = Reranker(backend=RerankerBackend.CROSS_ENCODER)
docs_local = local_reranker.rerank(query, docs_vector, top_n=5)
print('Local cross-encoder reranking:')
for d in docs_local:
    print(f'  [{d.metadata["rerank_score"]:.3f}] {d.page_content[:100]}...')

## Next: Episode 11 — Query Rewriting (HyDE + Multi-query)